# 🤖 Neural Network for Classification
This notebook walks through solving a classification problem using both **TensorFlow/Keras** and **PyTorch**.

## 📦 Install Dependencies (if needed)

In [ ]:
# Uncomment if running in Colab
# !pip install -q tensorflow torch scikit-learn pandas matplotlib


## 📂 Step 1: Load and Prepare the Dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Load sample dataset (binary classification)
data = load_breast_cancer()
X = data.data
y = data.target

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


## 🧠 Step 2A: Keras (TensorFlow) Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Build model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)


### 🔍 Evaluate Keras Model

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.3f}")


## 🔁 Step 2B: PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

# Define PyTorch model
class Net(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model_torch = Net(X_train.shape[1])
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_torch.parameters(), lr=0.001)


### 🔧 Train PyTorch Model

In [ ]:
for epoch in range(20):
    for xb, yb in train_dl:
        preds = model_torch(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


### 🔍 Evaluate PyTorch Model

In [ ]:
with torch.no_grad():
    preds = model_torch(X_test_tensor)
    predicted = (preds.numpy() > 0.5).astype("int")
    accuracy = (predicted == y_test.reshape(-1, 1)).mean()
    print(f"Test Accuracy (PyTorch): {accuracy:.3f}")
